# Paper Benchmarking Workflow

This notebook is the maintained entry point for the paper pipeline in this repository. Instead of reimplementing cross-framework runners inline, it orchestrates the checked-in paper scripts and inspects the generated CSV artifacts.


In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src" / "vamos").exists() and (candidate / "paper").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository root or a subdirectory inside it.")


REPO_ROOT = find_repo_root()
PAPER_DIR = REPO_ROOT / "paper"
EXPERIMENTS_DIR = REPO_ROOT / "experiments"
REPO_ROOT


## Inspect the benchmark tables already on disk


In [ ]:
benchmark_csv = EXPERIMENTS_DIR / "benchmark_paper.csv"
scaling_csv = EXPERIMENTS_DIR / "scaling_vectorization.csv"

bench_df = pd.read_csv(benchmark_csv) if benchmark_csv.exists() else pd.DataFrame()
scale_df = pd.read_csv(scaling_csv) if scaling_csv.exists() else pd.DataFrame()

bench_df.head(), scale_df.head()


## Rerun the benchmark producers

These commands use the checked-in paper scripts. Uncomment the ones you want to rerun; they can take a long time.


In [ ]:
# subprocess.run([sys.executable, str(PAPER_DIR / "01_run_paper_benchmark.py")], cwd=REPO_ROOT, check=True)
# subprocess.run([sys.executable, str(PAPER_DIR / "03_run_scaling_experiment.py")], cwd=REPO_ROOT, check=True)
# subprocess.run([sys.executable, str(PAPER_DIR / "31_run_zcat_all_tables.py")], cwd=REPO_ROOT, check=True)


## Rebuild the manuscript after the CSVs change


In [ ]:
# subprocess.run([sys.executable, str(PAPER_DIR / "08_compile_manuscript_pdf.py"), "--no-sync"], cwd=REPO_ROOT, check=True)


## Quick summaries


In [ ]:
if not bench_df.empty:
    summary = (
        bench_df.groupby(["framework", "suite"], dropna=False)["runtime_seconds"]
        .median()
        .reset_index()
        .sort_values(["suite", "runtime_seconds"])
    )
    summary.head(20)
